In [ ]:
import pandas as pd
import numpy as np
import re
from google.colab import files

In [ ]:
# Load the original dataset directly from the GitHub repo
url = "https://raw.githubusercontent.com/Saung210/budgetbites_recipe_chatbot/main/data/raw/original_dataset.csv"

df = pd.read_csv(url)
print(df.shape)
df.head()

(486, 12)


,Recipe_name,Category,Cuisine,Preptime,Tottime,Yield,Serving_size,Nutrition (per serving),Keywords,Decription,Recipe_steps,Ingredients
0,Banana Flambe Recipe,Dessert,European,5.0,10,2 Servings,1 bowl,Calories: 210 kcal\nProtein: 2 g\nFats: 8 g\nC...,"['Banana', 'Flambe', 'Recipe']",A rich and caramelized dessert made with banan...,1.Melt butter in a pan and add sliced bananas....,"Bananas, butter, sugar, cinnamon, rum (optiona..."
1,Greek Cheese Balls Recipe,Snack,European,10.0,20,8 Servings,1 bowl,Calories: 280 kcal\nProtein: 10 g\nFats: 18 g\...,"['Greek', 'Cheese', 'Balls', 'Recipe']","Crispy fried cheese balls with a soft, melty c...","1.Mix cheeses, herbs, and flour into a dough.\...","Feta cheese, mozzarella, flour, eggs, breadcru..."
2,Misal Pav Recipe,Lunch,Indian,10.0,30,8 Servings,1 bowl,Calories: 350 kcal\nProtein: 12 g\nFats: 12 g\...,"['Misal', 'Pav', 'Recipe']",A spicy and flavorful Maharashtrian dish made ...,1.Cook sprouted lentils with spices into a cur...,A spicy and flavorful Maharashtrian dish made ...
3,Oreo Cake Recipe,Dessert,European,10.0,15,5 Servings,1 bowl,Calories: 320 kcal\nProtein: 5 g\nFats: 15 g\n...,"['Oreo', 'Cake', 'Recipe']",A quick and easy chocolate cake made using cru...,1.Crush Oreo biscuits into a fine powder.\n2.M...,"Oreo biscuits, milk, sugar, baking powder, butter"
4,Grilled Fish in Lemon Butter Sauce Recipe,Lunch,European,10.0,20,2 Servings,1 bowl,Calories: 260 kcal\nProtein: 25 g\nFats: 14 g\...,"['Grilled', 'Fish', 'in', 'Lemon', 'Butter', '...",A light and healthy dish featuring perfectly g...,1.Season fish with salt and pepper.\n2.Grill u...,"Fish fillets, butter, lemon juice, garlic, sal..."


In [ ]:
# Check original column names
print(df.columns.tolist())

['Recipe_name', 'Category', 'Cuisine', 'Preptime', 'Tottime', 'Yield', 'Serving_size', 'Nutrition (per serving) ', 'Keywords', 'Decription', 'Recipe_steps', 'Ingredients']


In [ ]:
import pandas as pd

# Load the original dataset directly from the GitHub repo
url = "https://raw.githubusercontent.com/Saung210/budgetbites_recipe_chatbot/main/data/raw/original_dataset.csv"

df = pd.read_csv(url)

# Fix column names if needed
df = df.rename(columns={
    "Nutrition (per serving) ": "Nutrition_per_serving",
    "Nutrition (per serving)": "Nutrition_per_serving",
    "Decription": "Description"
})

# Make sure time columns are numeric
df["Preptime"] = pd.to_numeric(df["Preptime"], errors="coerce")
df["Tottime"] = pd.to_numeric(df["Tottime"], errors="coerce")

# Count rows where total time is less than prep time
bad_before = (df["Tottime"] < df["Preptime"]).sum()
print("Rows where Tottime < Preptime before fix:", bad_before)

# Fix: force total time to be at least prep time
mask = df["Tottime"] < df["Preptime"]
df.loc[mask, "Tottime"] = df.loc[mask, "Preptime"]

# Check again
bad_after = (df["Tottime"] < df["Preptime"]).sum()
print("Rows where Tottime < Preptime after fix:", bad_after)

# Save cleaned file
df.to_csv("recipe_dataset_fixed_time.csv", index=False)

print("Saved: recipe_dataset_fixed_time.csv")
df.head()

Rows where Tottime < Preptime before fix: 165
Rows where Tottime < Preptime after fix: 0
Saved: recipe_dataset_fixed_time.csv


,Recipe_name,Category,Cuisine,Preptime,Tottime,Yield,Serving_size,Nutrition_per_serving,Keywords,Description,Recipe_steps,Ingredients
0,Banana Flambe Recipe,Dessert,European,5.0,10,2 Servings,1 bowl,Calories: 210 kcal\nProtein: 2 g\nFats: 8 g\nC...,"['Banana', 'Flambe', 'Recipe']",A rich and caramelized dessert made with banan...,1.Melt butter in a pan and add sliced bananas....,"Bananas, butter, sugar, cinnamon, rum (optiona..."
1,Greek Cheese Balls Recipe,Snack,European,10.0,20,8 Servings,1 bowl,Calories: 280 kcal\nProtein: 10 g\nFats: 18 g\...,"['Greek', 'Cheese', 'Balls', 'Recipe']","Crispy fried cheese balls with a soft, melty c...","1.Mix cheeses, herbs, and flour into a dough.\...","Feta cheese, mozzarella, flour, eggs, breadcru..."
2,Misal Pav Recipe,Lunch,Indian,10.0,30,8 Servings,1 bowl,Calories: 350 kcal\nProtein: 12 g\nFats: 12 g\...,"['Misal', 'Pav', 'Recipe']",A spicy and flavorful Maharashtrian dish made ...,1.Cook sprouted lentils with spices into a cur...,A spicy and flavorful Maharashtrian dish made ...
3,Oreo Cake Recipe,Dessert,European,10.0,15,5 Servings,1 bowl,Calories: 320 kcal\nProtein: 5 g\nFats: 15 g\n...,"['Oreo', 'Cake', 'Recipe']",A quick and easy chocolate cake made using cru...,1.Crush Oreo biscuits into a fine powder.\n2.M...,"Oreo biscuits, milk, sugar, baking powder, butter"
4,Grilled Fish in Lemon Butter Sauce Recipe,Lunch,European,10.0,20,2 Servings,1 bowl,Calories: 260 kcal\nProtein: 25 g\nFats: 14 g\...,"['Grilled', 'Fish', 'in', 'Lemon', 'Butter', '...",A light and healthy dish featuring perfectly g...,1.Season fish with salt and pepper.\n2.Grill u...,"Fish fillets, butter, lemon juice, garlic, sal..."


In [ ]:
# Clean column names
df = df.rename(columns={
    "Nutrition (per serving) ": "Nutrition_per_serving",
    "Nutrition (per serving)": "Nutrition_per_serving",
    "Decription": "Description"
})

# standardize spacing/underscores in any remaining column names
df.columns = [str(c).strip() for c in df.columns]
print(df.columns.tolist())

['Recipe_name', 'Category', 'Cuisine', 'Preptime', 'Tottime', 'Yield', 'Serving_size', 'Nutrition_per_serving', 'Keywords', 'Description', 'Recipe_steps', 'Ingredients']


In [ ]:
# Keep the main columns used in your project
wanted_cols = [
    "Recipe_name",
    "Category",
    "Cuisine",
    "Preptime",
    "Tottime",
    "Yield",
    "Serving_size",
    "Nutrition_per_serving",
    "Keywords",
    "Description",
    "Recipe_steps",
    "Ingredients"
]

df = df[wanted_cols].copy()
print(df.shape)
df.head()

(486, 12)


,Recipe_name,Category,Cuisine,Preptime,Tottime,Yield,Serving_size,Nutrition_per_serving,Keywords,Description,Recipe_steps,Ingredients
0,Banana Flambe Recipe,Dessert,European,5.0,10,2 Servings,1 bowl,Calories: 210 kcal\nProtein: 2 g\nFats: 8 g\nC...,"['Banana', 'Flambe', 'Recipe']",A rich and caramelized dessert made with banan...,1.Melt butter in a pan and add sliced bananas....,"Bananas, butter, sugar, cinnamon, rum (optiona..."
1,Greek Cheese Balls Recipe,Snack,European,10.0,20,8 Servings,1 bowl,Calories: 280 kcal\nProtein: 10 g\nFats: 18 g\...,"['Greek', 'Cheese', 'Balls', 'Recipe']","Crispy fried cheese balls with a soft, melty c...","1.Mix cheeses, herbs, and flour into a dough.\...","Feta cheese, mozzarella, flour, eggs, breadcru..."
2,Misal Pav Recipe,Lunch,Indian,10.0,30,8 Servings,1 bowl,Calories: 350 kcal\nProtein: 12 g\nFats: 12 g\...,"['Misal', 'Pav', 'Recipe']",A spicy and flavorful Maharashtrian dish made ...,1.Cook sprouted lentils with spices into a cur...,A spicy and flavorful Maharashtrian dish made ...
3,Oreo Cake Recipe,Dessert,European,10.0,15,5 Servings,1 bowl,Calories: 320 kcal\nProtein: 5 g\nFats: 15 g\n...,"['Oreo', 'Cake', 'Recipe']",A quick and easy chocolate cake made using cru...,1.Crush Oreo biscuits into a fine powder.\n2.M...,"Oreo biscuits, milk, sugar, baking powder, butter"
4,Grilled Fish in Lemon Butter Sauce Recipe,Lunch,European,10.0,20,2 Servings,1 bowl,Calories: 260 kcal\nProtein: 25 g\nFats: 14 g\...,"['Grilled', 'Fish', 'in', 'Lemon', 'Butter', '...",A light and healthy dish featuring perfectly g...,1.Season fish with salt and pepper.\n2.Grill u...,"Fish fillets, butter, lemon juice, garlic, sal..."


In [ ]:
# Basic text cleaning
def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

text_cols = [
    "Recipe_name", "Category", "Cuisine", "Yield", "Serving_size",
    "Nutrition_per_serving", "Keywords", "Description",
    "Recipe_steps", "Ingredients"
]

for col in text_cols:
    df[col] = df[col].apply(clean_text)

df.head()

,Recipe_name,Category,Cuisine,Preptime,Tottime,Yield,Serving_size,Nutrition_per_serving,Keywords,Description,Recipe_steps,Ingredients
0,Banana Flambe Recipe,Dessert,European,5.0,10,2 Servings,1 bowl,Calories: 210 kcal Protein: 2 g Fats: 8 g Carb...,"['Banana', 'Flambe', 'Recipe']",A rich and caramelized dessert made with banan...,1.Melt butter in a pan and add sliced bananas....,"Bananas, butter, sugar, cinnamon, rum (optiona..."
1,Greek Cheese Balls Recipe,Snack,European,10.0,20,8 Servings,1 bowl,Calories: 280 kcal Protein: 10 g Fats: 18 g Ca...,"['Greek', 'Cheese', 'Balls', 'Recipe']","Crispy fried cheese balls with a soft, melty c...","1.Mix cheeses, herbs, and flour into a dough. ...","Feta cheese, mozzarella, flour, eggs, breadcru..."
2,Misal Pav Recipe,Lunch,Indian,10.0,30,8 Servings,1 bowl,Calories: 350 kcal Protein: 12 g Fats: 12 g Ca...,"['Misal', 'Pav', 'Recipe']",A spicy and flavorful Maharashtrian dish made ...,1.Cook sprouted lentils with spices into a cur...,A spicy and flavorful Maharashtrian dish made ...
3,Oreo Cake Recipe,Dessert,European,10.0,15,5 Servings,1 bowl,Calories: 320 kcal Protein: 5 g Fats: 15 g Car...,"['Oreo', 'Cake', 'Recipe']",A quick and easy chocolate cake made using cru...,1.Crush Oreo biscuits into a fine powder. 2.Mi...,"Oreo biscuits, milk, sugar, baking powder, butter"
4,Grilled Fish in Lemon Butter Sauce Recipe,Lunch,European,10.0,20,2 Servings,1 bowl,Calories: 260 kcal Protein: 25 g Fats: 14 g Ca...,"['Grilled', 'Fish', 'in', 'Lemon', 'Butter', '...",A light and healthy dish featuring perfectly g...,1.Season fish with salt and pepper. 2.Grill un...,"Fish fillets, butter, lemon juice, garlic, sal..."


In [ ]:
# Standardize Cuisine
def clean_cuisine(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().title()

df["Cuisine"] = df["Cuisine"].apply(clean_cuisine)
df["Cuisine"].value_counts(dropna=False)

,count
Cuisine,
Indian,328
European,111
American,27
Chinese,20


In [ ]:
# Convert time columns to numeric minutes
def extract_minutes(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    # direct numeric case
    try:
        return int(float(s))
    except:
        pass

    total = 0

    hour_match = re.search(r"(\d+(?:\.\d+)?)\s*(hour|hours|hr|hrs)", s)
    min_match = re.search(r"(\d+(?:\.\d+)?)\s*(minute|minutes|min|mins)", s)

    if hour_match:
        total += float(hour_match.group(1)) * 60
    if min_match:
        total += float(min_match.group(1))

    if total > 0:
        return int(round(total))

    return np.nan

df["Preptime"] = df["Preptime"].apply(extract_minutes)
df["Tottime"] = df["Tottime"].apply(extract_minutes)

df[["Preptime", "Tottime"]].head()

,Preptime,Tottime
0,5,10
1,10,20
2,10,30
3,10,15
4,10,20


In [ ]:
# Flag bad time rows
df["Flag_bad_time"] = (
    df["Preptime"].isna() |
    df["Tottime"].isna() |
    (df["Preptime"] < 0) |
    (df["Tottime"] < 0) |
    (df["Tottime"] < df["Preptime"])
)

df["Flag_bad_time"].value_counts()

,count
Flag_bad_time,
False,486


In [ ]:
# Clean Ingredients
def clean_ingredients(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # remove bracket/list formatting
    s = s.replace("[", "").replace("]", "")
    s = s.replace("'", "").replace('"', "")

    # make separators consistent
    s = s.replace(";", ",")
    s = re.sub(r"\s*,\s*", ", ", s)
    s = re.sub(r"\s+", " ", s).strip(" ,")

    # capitalize first letter only
    if len(s) > 0:
        s = s[0].upper() + s[1:]

    return s

df["Ingredients"] = df["Ingredients"].apply(clean_ingredients)

df[["Recipe_name", "Ingredients"]].head()

,Recipe_name,Ingredients
0,Banana Flambe Recipe,"Bananas, butter, sugar, cinnamon, rum (optiona..."
1,Greek Cheese Balls Recipe,"Feta cheese, mozzarella, flour, eggs, breadcru..."
2,Misal Pav Recipe,A spicy and flavorful Maharashtrian dish made ...
3,Oreo Cake Recipe,"Oreo biscuits, milk, sugar, baking powder, butter"
4,Grilled Fish in Lemon Butter Sauce Recipe,"Fish fillets, butter, lemon juice, garlic, sal..."


In [ ]:
# Flag suspicious ingredient rows
same_as_description = (
    df["Ingredients"].fillna("").str.lower().str.strip() ==
    df["Description"].fillna("").str.lower().str.strip()
)

too_long = df["Ingredients"].fillna("").str.len() > 180
no_comma = ~df["Ingredients"].fillna("").str.contains(",")

df["Flag_bad_ingredients"] = same_as_description | too_long | no_comma

df["Flag_bad_ingredients"].value_counts()

,count
Flag_bad_ingredients,
False,483
True,3


In [ ]:
# Clean recipe steps
def clean_steps(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # put numbered steps on separate lines
    s = re.sub(r"(?<!\n)(\d+\.)", r"\n\1", s).strip()
    s = re.sub(r"\n+", "\n", s)

    return s

df["Recipe_steps"] = df["Recipe_steps"].apply(clean_steps)

df["Flag_bad_steps"] = (
    df["Recipe_steps"].isna() |
    (df["Recipe_steps"].fillna("").str.len() < 20)
)

df["Flag_bad_steps"].value_counts()

,count
Flag_bad_steps,
False,486


In [ ]:
# Extract nutrition into separate columns
def extract_nutrition(text, patterns):
    if pd.isna(text):
        return np.nan

    s = str(text).lower().replace("\n", " ")

    for pattern in patterns:
        match = re.search(pattern, s)
        if match:
            try:
                return float(match.group(1))
            except:
                return np.nan
    return np.nan

df["Calories"] = df["Nutrition_per_serving"].apply(
    lambda x: extract_nutrition(x, [r"calories:\s*([\d\.]+)"])
)

df["Protein_g"] = df["Nutrition_per_serving"].apply(
    lambda x: extract_nutrition(x, [r"protein:\s*([\d\.]+)"])
)

df["Fats_g"] = df["Nutrition_per_serving"].apply(
    lambda x: extract_nutrition(x, [r"fats?:\s*([\d\.]+)"])
)

df["Carbs_g"] = df["Nutrition_per_serving"].apply(
    lambda x: extract_nutrition(x, [r"carbs?:\s*([\d\.]+)"])
)

df["Flag_bad_nutrition"] = (
    df["Calories"].isna() |
    df["Protein_g"].isna() |
    df["Fats_g"].isna() |
    df["Carbs_g"].isna()
)

df[["Calories", "Protein_g", "Fats_g", "Carbs_g"]].head()

,Calories,Protein_g,Fats_g,Carbs_g
0,210.0,2.0,8.0,35.0
1,280.0,10.0,18.0,20.0
2,350.0,12.0,12.0,50.0
3,320.0,5.0,15.0,42.0
4,260.0,25.0,14.0,5.0


In [ ]:
# Remove exact duplicates
df = df.drop_duplicates()

# Create review file with flagged rows
review_df = df[
    df["Flag_bad_time"] |
    df["Flag_bad_ingredients"] |
    df["Flag_bad_steps"] |
    df["Flag_bad_nutrition"]
].copy()

print("Total rows:", len(df))
print("Rows needing review:", len(review_df))

Total rows: 486
Rows needing review: 3


In [ ]:
# Save outputs
cleaned_file_xlsx = "recipe_dataset_cleaned.xlsx"
cleaned_file_csv = "recipe_dataset_cleaned.csv"
review_file_xlsx = "recipe_dataset_review_needed.xlsx"

df.to_excel(cleaned_file_xlsx, index=False)
df.to_csv(cleaned_file_csv, index=False)
review_df.to_excel(review_file_xlsx, index=False)

print("Saved all files.")

Saved all files.


In [ ]:
# Download the cleaned files
files.download("recipe_dataset_cleaned.xlsx")
files.download("recipe_dataset_cleaned.csv")
files.download("recipe_dataset_review_needed.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>